In [1]:
%matplotlib inline

%load_ext autoreload
%autoreload 2

In [2]:
import os
import sys
from os.path import exists

sys.path.append('../..')

In [3]:
import pylab as plt
import pandas as pd
import numpy as np
from loguru import logger
import seaborn as sns

from stable_baselines3 import PPO, DQN

In [4]:
from vimms.Common import POSITIVE, set_log_level_warning, load_obj, save_obj
from vimms.ChemicalSamplers import UniformRTAndIntensitySampler, GaussianChromatogramSampler, UniformMZFormulaSampler, \
    MZMLFormulaSampler, MZMLRTandIntensitySampler, MZMLChromatogramSampler
from vimms.Noise import UniformSpikeNoise
from vimms.Evaluation import evaluate_real
from vimms.Chemicals import ChemicalMixtureFromMZML
from vimms.Roi import RoiBuilderParams, SmartRoiParams

from mass_spec_utils.data_import.mzmine import load_picked_boxes

from vimms_gym.env import DDAEnv
from vimms_gym.chemicals import generate_chemicals
from vimms_gym.evaluation import evaluate, run_method
from vimms_gym.common import METHOD_RANDOM, METHOD_FULLSCAN, METHOD_TOPN, METHOD_PPO, METHOD_DQN

/opt/anaconda3/envs/vimms-gym/lib/python3.9/site-packages/statsmodels/compat/pandas.py:61: FutureWarning: pandas.Int64Index is deprecated and will be removed from pandas in a future version. Use pandas.Index with the appropriate dtype instead.
  from pandas import Int64Index as NumericIndex
/opt/anaconda3/envs/vimms-gym/lib/python3.9/site-packages/psims/mzmlb/writer.py:15: UserWarning: hdf5plugin is missing! Only the slower GZIP compression scheme will be available! Please install hdf5plugin to be able to use Blosc.
  warnings.warn(


# 1. Parameters

In [5]:
# n_chemicals = (200, 500)
# mz_range = (100, 600)
# rt_range = (0, 300)
# intensity_range = (1E5, 1E10)

In [6]:
n_chemicals = (2000, 5000)
mz_range = (100, 600)
rt_range = (200, 1000)
intensity_range = (1E4, 1E10)

In [7]:
min_mz = mz_range[0]
max_mz = mz_range[1]
min_rt = rt_range[0]
max_rt = rt_range[1]
min_log_intensity = np.log(intensity_range[0])
max_log_intensity = np.log(intensity_range[1])

In [8]:
isolation_window = 0.7
N = 10
rt_tol = 120
exclusion_t_0 = 15
mz_tol = 10
min_ms1_intensity = 5000
ionisation_mode = POSITIVE

enable_spike_noise = True
noise_density = 0.1
noise_max_val = 1E3

In [9]:
mz_sampler = UniformMZFormulaSampler(min_mz=min_mz, max_mz=max_mz)
ri_sampler = UniformRTAndIntensitySampler(min_rt=min_rt, max_rt=max_rt,
                                          min_log_intensity=min_log_intensity,
                                          max_log_intensity=max_log_intensity)
cr_sampler = GaussianChromatogramSampler()
samplers = {
    'mz': mz_sampler,
    'rt_intensity': ri_sampler,
    'chromatogram': cr_sampler
}

In [10]:
params = {
    'chemical_creator': {
        'mz_range': mz_range,
        'rt_range': rt_range,
        'intensity_range': intensity_range,
        'n_chemicals': n_chemicals,
        'mz_sampler': mz_sampler,
        'ri_sampler': ri_sampler,
        'cr_sampler': cr_sampler,
    },
    'noise': {
        'enable_spike_noise': enable_spike_noise,
        'noise_density': noise_density,
        'noise_max_val': noise_max_val,
        'mz_range': mz_range
    },
    'env': {
        'ionisation_mode': ionisation_mode,
        'rt_range': rt_range,
        'isolation_window': isolation_window,
        'mz_tol': mz_tol,
        'rt_tol': rt_tol,
    }
}

In [11]:
max_peaks = 200
in_dir = 'results'

In [12]:
n_eval_episodes = 1
deterministic = True

# 2. Evaluation

#### Generate some chemical sets

In [13]:
set_log_level_warning()

1

In [14]:
eval_dir = 'optimise_baselines'
method = METHOD_TOPN

In [15]:
chemical_creator_params = params['chemical_creator']

chem_list = []
for i in range(n_eval_episodes):
    print(i)
    chems = generate_chemicals(chemical_creator_params)
    chem_list.append(chems)

0


#### Run different methods

In [16]:
for chems in chem_list:
    print(len(chems))

2937


In [17]:
max_peaks

200

In [18]:
out_dir = eval_dir
in_dir, out_dir

('results', 'optimise_baselines')

#### Compare to Top-10

In [23]:
env_name = 'DDAEnv'
model_name = 'PPO'
intensity_threshold = 0.5

In [24]:
rt_tols = [15, 30, 60, 120, 240, 300]
Ns = [5, 10, 15, 20, 25]

In [25]:
topN_res = {}
for rt_tol in rt_tols:
    for N in Ns:

        effective_rt_tol = rt_tol
        copy_params = dict(params)        
        copy_params['env']['rt_tol'] = effective_rt_tol

        banner = 'method = %s max_peaks = %d N = %d rt_tol = %d' % (method, max_peaks, N, effective_rt_tol)
        print(banner)
        print()

        if method == METHOD_PPO:
            fname = os.path.join(in_dir, '%s_%s.zip' % (env_name, model_name))
            model = PPO.load(fname)
        elif method == METHOD_DQN:
            fname = os.path.join(in_dir, '%s_%s.zip' % (env_name, model_name))
            model = DQN.load(fname)
        else:
            model = None

        episodic_results = run_method(env_name, copy_params, max_peaks, chem_list, method, out_dir, 
                                      N=N, min_ms1_intensity=min_ms1_intensity, model=model,
                                      print_eval=True, print_reward=False, intensity_threshold=intensity_threshold)
        eval_results = [er.eval_res for er in episodic_results][0]

        key = (N, rt_tol)
        topN_res[key] = eval_results
        print()

method = topN max_peaks = 200 N = 5 rt_tol = 15

{'coverage_prop': '0.433', 'intensity_prop': '0.349', 'ms1/ms2 ratio': '0.202', 'efficiency': '0.447', 'TP': '969', 'FP': '114', 'FN': '1854', 'precision': '0.895', 'recall': '0.343', 'f1': '0.496'}

method = topN max_peaks = 200 N = 10 rt_tol = 15

{'coverage_prop': '0.468', 'intensity_prop': '0.381', 'ms1/ms2 ratio': '0.104', 'efficiency': '0.415', 'TP': '1052', 'FP': '125', 'FN': '1760', 'precision': '0.894', 'recall': '0.374', 'f1': '0.527'}

method = topN max_peaks = 200 N = 15 rt_tol = 15

{'coverage_prop': '0.491', 'intensity_prop': '0.405', 'ms1/ms2 ratio': '0.070', 'efficiency': '0.411', 'TP': '1108', 'FP': '136', 'FN': '1693', 'precision': '0.891', 'recall': '0.396', 'f1': '0.548'}

method = topN max_peaks = 200 N = 20 rt_tol = 15

{'coverage_prop': '0.498', 'intensity_prop': '0.412', 'ms1/ms2 ratio': '0.054', 'efficiency': '0.406', 'TP': '1128', 'FP': '132', 'FN': '1677', 'precision': '0.895', 'recall': '0.402', 'f1': '0.555'}

In [27]:
topN_res

{(5, 15): {'coverage_prop': '0.433',
  'intensity_prop': '0.349',
  'ms1/ms2 ratio': '0.202',
  'efficiency': '0.447',
  'TP': '969',
  'FP': '114',
  'FN': '1854',
  'precision': '0.895',
  'recall': '0.343',
  'f1': '0.496'},
 (10, 15): {'coverage_prop': '0.468',
  'intensity_prop': '0.381',
  'ms1/ms2 ratio': '0.104',
  'efficiency': '0.415',
  'TP': '1052',
  'FP': '125',
  'FN': '1760',
  'precision': '0.894',
  'recall': '0.374',
  'f1': '0.527'},
 (15, 15): {'coverage_prop': '0.491',
  'intensity_prop': '0.405',
  'ms1/ms2 ratio': '0.070',
  'efficiency': '0.411',
  'TP': '1108',
  'FP': '136',
  'FN': '1693',
  'precision': '0.891',
  'recall': '0.396',
  'f1': '0.548'},
 (20, 15): {'coverage_prop': '0.498',
  'intensity_prop': '0.412',
  'ms1/ms2 ratio': '0.054',
  'efficiency': '0.406',
  'TP': '1128',
  'FP': '132',
  'FN': '1677',
  'precision': '0.895',
  'recall': '0.402',
  'f1': '0.555'},
 (25, 15): {'coverage_prop': '0.513',
  'intensity_prop': '0.428',
  'ms1/ms2 rati

In [26]:
method_eval_results = {
    method: topN_res
}

#### Test classic controllers in ViMMS

In [28]:
from vimms.MassSpec import IndependentMassSpectrometer
from vimms.Controller import TopNController, TopN_SmartRoiController, WeightedDEWController
from vimms.Environment import Environment

In [32]:
spike_noise = None
if enable_spike_noise:
    noise_params = params['noise']
    noise_density = noise_params['noise_density']
    noise_max_val = noise_params['noise_max_val']
    noise_min_mz = noise_params['mz_range'][0]
    noise_max_mz = noise_params['mz_range'][1]
    spike_noise = UniformSpikeNoise(noise_density, noise_max_val, min_mz=noise_min_mz,
                                    max_mz=noise_max_mz)

Run Top-N Controller

In [31]:
method = 'TopN_Controller'
print('method = %s' % method)
print()

chems = chem_list[0]
res = {}
for rt_tol in rt_tols:
    for N in Ns:

        effective_rt_tol = rt_tol
        mass_spec = IndependentMassSpectrometer(ionisation_mode, chems, spike_noise=spike_noise)
        controller = TopNController(ionisation_mode, N, isolation_window, mz_tol, rt_tol,
                                    min_ms1_intensity)
        env = Environment(mass_spec, controller, min_rt, max_rt, progress_bar=False, out_dir=out_dir,
                          out_file='%s_%d.mzML' % (method, i), save_eval=True)
        env.run()
        eval_res = evaluate(env, intensity_threshold)
        key = (N, rt_tol)
        print(N, rt_tol, eval_res)
        res[key] = eval_res

method_eval_results[method] = res

method = TopN_Controller

5 15 {'coverage_prop': '0.433', 'intensity_prop': '0.347', 'ms1/ms2 ratio': '0.202', 'efficiency': '0.447', 'TP': '964', 'FP': '125', 'FN': '1848', 'precision': '0.885', 'recall': '0.343', 'f1': '0.494'}
10 15 {'coverage_prop': '0.508', 'intensity_prop': '0.407', 'ms1/ms2 ratio': '0.104', 'efficiency': '0.450', 'TP': '1161', 'FP': '136', 'FN': '1640', 'precision': '0.895', 'recall': '0.414', 'f1': '0.567'}
15 15 {'coverage_prop': '0.547', 'intensity_prop': '0.449', 'ms1/ms2 ratio': '0.071', 'efficiency': '0.459', 'TP': '1258', 'FP': '142', 'FN': '1537', 'precision': '0.899', 'recall': '0.450', 'f1': '0.600'}
20 15 {'coverage_prop': '0.573', 'intensity_prop': '0.479', 'ms1/ms2 ratio': '0.054', 'efficiency': '0.466', 'TP': '1306', 'FP': '156', 'FN': '1475', 'precision': '0.893', 'recall': '0.470', 'f1': '0.616'}
25 15 {'coverage_prop': '0.539', 'intensity_prop': '0.449', 'ms1/ms2 ratio': '0.045', 'efficiency': '0.431', 'TP': '1246', 'FP': '139', 'FN': '1552', 'p

Run SmartROI Controller

TO FINISH BELOW

In [45]:
alphas = [2, 3, 5, 10, 1E3, 1E6]
betas = [0, 0.1, 0.5, 1, 5]
smartroi_N = 20
smartroi_dew = 15

In [46]:
method = 'SmartROI_Controller'
print('method = %s' % method)
print()

chems = chem_list[0]
res = {}
for alpha in alphas:
    for beta in betas:

        mass_spec = IndependentMassSpectrometer(ionisation_mode, chems, spike_noise=spike_noise)
        
        roi_params = RoiBuilderParams(min_roi_intensity=500, min_roi_length=0)    
        smartroi_params = SmartRoiParams(intensity_increase_factor=alpha, drop_perc=beta/100.0)
        controller = TopN_SmartRoiController(ionisation_mode, isolation_window, smartroi_N, mz_tol, smartroi_dew,
                                    min_ms1_intensity, roi_params, smartroi_params)

        env = Environment(mass_spec, controller, min_rt, max_rt, progress_bar=False, out_dir=out_dir,
                          out_file='%s_%d.mzML' % (method, i), save_eval=True)
        env.run()
        eval_res = evaluate(env, intensity_threshold)
        key = (N, rt_tol)
        print(alpha, beta, eval_res)
        res[key] = eval_res

method_eval_results[method] = res

method = SmartROI_Controller

2 0 {'coverage_prop': '0.762', 'intensity_prop': '0.638', 'ms1/ms2 ratio': '0.058', 'efficiency': '0.624', 'TP': '1840', 'FP': '143', 'FN': '954', 'precision': '0.928', 'recall': '0.659', 'f1': '0.770'}
2 0.1 {'coverage_prop': '0.762', 'intensity_prop': '0.638', 'ms1/ms2 ratio': '0.058', 'efficiency': '0.624', 'TP': '1843', 'FP': '141', 'FN': '953', 'precision': '0.929', 'recall': '0.659', 'f1': '0.771'}
2 0.5 {'coverage_prop': '0.761', 'intensity_prop': '0.638', 'ms1/ms2 ratio': '0.058', 'efficiency': '0.623', 'TP': '1839', 'FP': '141', 'FN': '957', 'precision': '0.929', 'recall': '0.658', 'f1': '0.770'}
2 1 {'coverage_prop': '0.761', 'intensity_prop': '0.638', 'ms1/ms2 ratio': '0.058', 'efficiency': '0.624', 'TP': '1839', 'FP': '142', 'FN': '956', 'precision': '0.928', 'recall': '0.658', 'f1': '0.770'}
2 5 {'coverage_prop': '0.722', 'intensity_prop': '0.605', 'ms1/ms2 ratio': '0.058', 'efficiency': '0.591', 'TP': '1748', 'FP': '127', 'FN': '1062', 'preci

Run WeightedDEW Controller

In [54]:
t0s = [1, 3, 10, 15, 30, 60]
t1s = [15, 60, 120, 240, 360, 3600]
weighteddew_N = 20

In [55]:
method = 'WeightedDEW_Controller'
print('method = %s' % method)
print()

chems = chem_list[0]
res = {}
for t0 in t0s:
    for t1 in t1s:

        if t0 > t1:
            print('Invalid combination')
            continue
        
        mass_spec = IndependentMassSpectrometer(ionisation_mode, chems, spike_noise=spike_noise)
        
        controller = WeightedDEWController(ionisation_mode, weighteddew_N, isolation_window, mz_tol, t1,
                                    min_ms1_intensity, exclusion_t_0=t0)
        
        env = Environment(mass_spec, controller, min_rt, max_rt, progress_bar=False, out_dir=out_dir,
                          out_file='%s_%d.mzML' % (method, i), save_eval=True)
        env.run()
        eval_res = evaluate(env, intensity_threshold)
        key = (t0, t1)
        print(t0, t1, eval_res)
        res[key] = eval_res
        
method_eval_results[method] = res

method = WeightedDEW_Controller

1 15 {'coverage_prop': '0.317', 'intensity_prop': '0.278', 'ms1/ms2 ratio': '0.051', 'efficiency': '0.256', 'TP': '720', 'FP': '63', 'FN': '2154', 'precision': '0.920', 'recall': '0.251', 'f1': '0.394'}
1 60 {'coverage_prop': '0.402', 'intensity_prop': '0.343', 'ms1/ms2 ratio': '0.051', 'efficiency': '0.325', 'TP': '899', 'FP': '93', 'FN': '1945', 'precision': '0.906', 'recall': '0.316', 'f1': '0.469'}
1 120 {'coverage_prop': '0.442', 'intensity_prop': '0.362', 'ms1/ms2 ratio': '0.051', 'efficiency': '0.358', 'TP': '948', 'FP': '150', 'FN': '1839', 'precision': '0.863', 'recall': '0.340', 'f1': '0.488'}
1 240 {'coverage_prop': '0.473', 'intensity_prop': '0.372', 'ms1/ms2 ratio': '0.051', 'efficiency': '0.383', 'TP': '941', 'FP': '246', 'FN': '1750', 'precision': '0.793', 'recall': '0.350', 'f1': '0.485'}
1 360 {'coverage_prop': '0.494', 'intensity_prop': '0.378', 'ms1/ms2 ratio': '0.051', 'efficiency': '0.400', 'TP': '928', 'FP': '324', 'FN': '1685', 'p